# SQL Analysis

## Objective

The objective of this notebook is to analyze vehicle motion sensor data using SQL queries. The analysis focuses on understanding driving behavior patterns through aggregation and window functions.

In [8]:
import sqlite3
import pandas as pd

connection = sqlite3.connect("C:/Users/htupk/OneDrive/Desktop/DriveIntel/sql/driveintel.db")

In [9]:
query = """
SELECT
    "Target(Class)" AS Driver_Class,

    AVG(ABS(AccX)+ABS(AccY)+ABS(AccZ)) AS Avg_Acceleration,

    AVG(ABS(GyroX)+ABS(GyroY)+ABS(GyroZ)) AS Avg_Rotation

FROM sensor_readings

GROUP BY "Target(Class)"

ORDER BY Driver_Class;
"""

pd.read_sql(query, connection)

,Driver_Class,Avg_Acceleration,Avg_Rotation
0,1,1.426036,8.362202
1,2,1.392614,17.123887
2,3,1.417960,15.939302
3,4,1.408733,7.764517


### Observation

The query summarizes the average motion characteristics for each driving behavior class. Comparing average acceleration and rotation helps identify whether certain classes exhibit more aggressive movement patterns, providing an initial understanding before statistical testing and machine learning.

## Business Question 2

**Question:**

How does the rolling average of forward acceleration (AccX) change over time within each driving behavior class?

This analysis uses a SQL window function to calculate a moving average over the current row and the previous 10 sensor readings.

In [10]:
query = """
SELECT
    ROW_NUMBER() OVER() AS Reading_Number,

    "Target(Class)" AS Driver_Class,

    AccX,

    AVG(AccX) OVER (
        PARTITION BY "Target(Class)"
        ORDER BY rowid
        ROWS BETWEEN 10 PRECEDING AND CURRENT ROW
    ) AS Rolling_Avg_AccX

FROM sensor_readings

LIMIT 30;
"""

rolling_df = pd.read_sql(query, connection)

rolling_df

,Reading_Number,Driver_Class,AccX,Rolling_Avg_AccX
0,1,1,0.162598,0.162598
1,2,1,0.175781,0.169189
2,3,1,0.322754,0.220378
3,4,1,0.480225,0.285339
4,5,1,0.426025,0.313477
5,6,1,0.383789,0.325195
6,7,1,0.404785,0.336565
7,8,1,0.346924,0.337860
8,9,1,0.276611,0.331055
9,10,1,0.158936,0.313843


### Observation

The rolling average smooths short-term fluctuations in acceleration by averaging the current reading with the previous ten readings within the same driving behavior class. This provides a clearer view of overall driving trends while reducing the impact of momentary spikes or noise.

In [11]:
query = """
SELECT
    "Target(Class)" AS Driver_Class,

    MAX(ABS(GyroX)) AS Max_GyroX,

    MAX(ABS(GyroY)) AS Max_GyroY,

    MAX(ABS(GyroZ)) AS Max_GyroZ

FROM sensor_readings

GROUP BY "Target(Class)"

ORDER BY Driver_Class;
"""

pd.read_sql(query, connection)

,Driver_Class,Max_GyroX,Max_GyroY,Max_GyroZ
0,1,14.648855,13.564886,13.671756
1,2,12.778626,11.992366,50.259542
2,3,14.946565,16.793893,45.442748
3,4,13.358779,12.984733,11.625954


### Observation

This query identifies the maximum rotational motion observed in each driving behavior class. Large gyroscope values may indicate sharp turns, rapid steering changes, or aggressive cornering, which can be useful indicators of risky driving behavior.

In [12]:
connection.close()

print("Database connection closed.")

Database connection closed.
